# Refresh — re-read saved videos / channels from the YouTube API (step 1 of the daily job)

Runs the app's `refresh_daily.py` from `src/app` (deployed with the bundle, so the job and the app share one copy of the code) on serverless. It writes the latest counters into Lakebase Postgres `public.videos` / `public.channels`, so Bronze → Silver get one snapshot per video / channel per day.

**Empty-source guard:** `refresh_daily.py` reads everything first and raises before writing if the API returns 0 rows or more than 20% zero counters, so this task fails and the rest of the job does not run (no day of zeros).

**Secrets (scope `youtube`):** `pg_password` (Postgres role `admin`), `api_key` (YouTube Data API v3).

Widget `dry_run` = `true` calls the API and runs the guard but writes nothing.


In [ ]:
%pip install --quiet "psycopg[binary]==3.2.10" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

os.environ.update({
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    "YOUTUBE_API_KEY": dbutils.secrets.get("youtube", "api_key"),
})

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
sys.path.insert(0, app_dir)
print(f"App code: {app_dir} · dry_run = {DRY_RUN}")


In [ ]:
# --- Refresh (raises EmptySourceError before writing if the API answer looks wrong) ---
import refresh_daily

counts = refresh_daily.main(["--dry-run"] if DRY_RUN else [])
print(f"✅ Refresh complete: {counts}")
